In [1]:
import h5py
import numpy as np
import os
import pandas as pd
import pathlib
import pickle
import scipy.stats as stats
import soundfile as sf
import soxr
#! change below to spatial_attn_lighting if want to use with modular 
import src.spatial_attn_lightning as attn_tracking_lightning
import torch
import yaml

from argparse import ArgumentParser
from corpus.speaker_room_dataset import SpeakerRoomDataset
from pytorch_lightning import Trainer, seed_everything
from tqdm.auto import tqdm

os.environ["HDF5_USE_FILE_LOCKING"] = "FALSE"
seed_everything(1)

Global seed set to 1


1

In [2]:
model_name = 'word_task_mixed_cue_large'
checkpoint_path = '/om2/user/imgriff/projects/Auditory-Attention/attn_cue_models/word_task_mixed_cue_large_architecture_v04/checkpoints/epoch=0-step=1000-v8.ckpt'
cue_type = 'both'

config = yaml.load(open('/om2/user/imgriff/projects/Auditory-Attention/config/binaural_attn/word_task_mixed_cue_large_architecture_v04.yml', 'r'), Loader=yaml.FullLoader)
config['num_workers'] = 10
#! change batch size
config['hparas']['batch_size'] = 1 # config['data']['loader']['batch_size'] // args.gpus
config['noise_kwargs']['low_snr'] = 0
config['noise_kwargs']['high_snr'] = 0

In [3]:
target_loc = (0,0)
distract_loc = (90,0)

model = attn_tracking_lightning.BinauralAttentionModule.load_from_checkpoint(checkpoint_path=checkpoint_path, config=config).cuda()
audio_transforms = model.audio_transforms

num_classes={'num_words': 800}
Model performing word task
cochlea_filt {'sr': 50000, 'env_sr': 10000, 'n_channels': 40, 'low_lim': 40, 'use_pad': True, 'binaural': True, 'rep_on_gpu': True, 'center_crop': True, 'out_dur': 2, 'impulse_len': 0.25, 'env_extraction_type': 'Half-wave Rectification', 'downsampling_type': 'TorchTransformsResample', 'downsampling_kwargs': {'lowpass_filter_width': 64, 'rolloff': 0.9475937167399596, 'resampling_method': 'kaiser_window', 'beta': 14.769656459379492}} coch_p3 {'scale': 1, 'offset': 1e-07, 'clip_value': 5, 'power': 0.3}
center_crop=True
binaural=True
Binaural cochleagram
using FIR cochleagram


In [4]:
dataset = SpeakerRoomDataset('/om2/user/rphess/Auditory-Attention/final_binaural_manifest.pkl', '/om2/user/msaddler/spatial_audio_pipeline/assets/swc/manifest_all_words.pdpkl', 'both')

In [5]:

def spatialize(words, ir):
    """Uses pytorch to convolve all sounds in words with 2 channel IR given in ir"""
    n_words = words.shape[0]
    words_padded = [torch.nn.functional.pad(word, (ir.shape[0] - 1, 0)) for word in words]
    ir = ir.T.unsqueeze(1)
    words_padded = torch.stack(words_padded)
    spatialized = torch.nn.functional.conv1d(words_padded.view(n_words, 1, -1).cuda(), ir.cuda()).cuda()
    return spatialized

In [6]:
dataloader = torch.utils.data.DataLoader(dataset, batch_size=30, shuffle=False, num_workers=config['num_workers'])

In [7]:
#TODO change this account for specific room
new_room_manifest = pd.read_pickle('/om2/user/msaddler/spatial_audio_pipeline/assets/brir/mit_bldg46room1004/manifest_brir.pdpkl')
only14_manifest = new_room_manifest[new_room_manifest['src_dist'] == 1.4]
ir_dict = dict()
for loc in ['target', 'distractor', 'cue']:
    if loc == 'target':
        coords = target_loc
    elif loc == 'distractor':
        coords = distract_loc
    else:
        if cue_type == 'voice':
            coords = (0,0)
        else:
            coords = target_loc
    df_row = only14_manifest[(only14_manifest['src_azim'] == coords[0]) & (only14_manifest['src_elev'] == coords[1])]
    h5_fn = f'/om2/user/msaddler/spatial_audio_pipeline/assets/brir/mit_bldg46room1004/room000{df_row["index_room"].values[0]}.hdf5'
    index_brir = df_row['index_brir'].values[0]
    sr_src = df_row['sr'].values[0]
    with h5py.File(h5_fn, 'r') as f:
        brir = f['brir'][index_brir]
    new_sr = 50000
    brir = soxr.resample(brir.astype(np.float32), sr_src, new_sr)
    ir_dict[loc] = brir

tar_brir = torch.from_numpy(ir_dict['target'])
tar_brir = torch.flip(tar_brir, dims=[0])
dist_brir = torch.from_numpy(ir_dict['distractor'])
dist_brir = torch.flip(dist_brir, dims=[0])
cue_brir = torch.from_numpy(ir_dict['cue'])
cue_brir = torch.flip(cue_brir, dims=[0])

In [9]:
accuracies = []
confusions = []
for batch in tqdm(dataloader):
    cue, fg, bg, label, confusion = batch
    cue_test = cue

    cue = np.array(spatialize(cue.cuda(), cue_brir.cuda()).cpu()[:, :, 12500:137500])
    foreground = np.array(spatialize(fg.cuda(), tar_brir.cuda()).cpu()[:, :, 12500:137500])
    background = np.array(spatialize(bg.cuda(), dist_brir.cuda()).cpu()[:, :, 12500:137500])

    cue = audio_transforms(cue, None)[0]
    mixture = audio_transforms(foreground, background)[0]

    cue = cue.cuda()
    mixture = mixture.cuda()
    logits = model(cue, mixture, None)

    preds = logits.softmax(dim=-1).argmax(dim=-1).cpu().detach().numpy().astype('int')
    true_word = label.numpy().astype('int')
    con_word = confusion.numpy().astype('int')
    accuracy = (preds == true_word).astype('int')
    cons = (preds == con_word).astype('int')
    accuracies.append(accuracy)
    confusions.append(cons)

  0%|          | 0/106 [00:01<?, ?it/s]

Making big manifest that's passed in

In [26]:
final_df = pd.DataFrame(index = range(1578 * 2), columns = ['src_ix', 'client_id', 'corpus', 'gender', 'gender_int', 'split', 'split_int', 'word', 'cue_word', 'cue_src_ix', 'cue_client_id', 'bg_word', 'bg_src_ix', 'bg_client_id', 'bg_gender'])

In [29]:
male_df = df[df['gender'] == 'male']
female_df = df[df['gender'] == 'female']

for condition in ['m_m', 'm_f', 'f_m', 'f_f']:
    if condition[0] == 'm':
        tar_df = male_df
    else:
        tar_df = female_df
    if condition[2] == 'm':
        cond_num = 1578
        dist_df = male_df
    else:
        cond_num = 0
        dist_df = female_df
    for i, row in tar_df.iterrows():
        final_df.loc[cond_num + i, 'src_ix'] = row['src_ix']
        final_df.loc[cond_num + i, 'client_id'] = row['client_id']
        final_df.loc[cond_num + i, 'corpus'] = row['corpus']
        final_df.loc[cond_num + i, 'gender'] = row['gender']
        final_df.loc[cond_num + i, 'gender_int'] = row['gender_int']
        final_df.loc[cond_num + i, 'split'] = row['split']
        final_df.loc[cond_num + i, 'split_int'] = row['split_int']
        final_df.loc[cond_num + i, 'word'] = row['word']
        final_df.loc[cond_num + i, 'cue_word'] = row['cue_word']
        final_df.loc[cond_num + i, 'cue_src_ix'] = row['cue_src_ix']
        final_df.loc[cond_num + i, 'cue_client_id'] = row['cue_client_id']
        distractor = dist_df[dist_df['client_id'] != row['client_id']].sample(1)
        final_df.loc[cond_num + i, 'bg_word'] = distractor['word'].values[0]
        final_df.loc[cond_num + i, 'bg_src_ix'] = distractor['src_ix'].values[0]
        final_df.loc[cond_num + i, 'bg_client_id'] = distractor['client_id'].values[0]
        final_df.loc[cond_num + i, 'bg_gender'] = distractor['gender'].values[0]

In [35]:
with open('final_binaural_manifest.pkl', 'wb') as f:
    pickle.dump(final_df, f)